<a href="https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Investigacion_Avanzada/Ondas_Solitones.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Investigación Avanzada: Ondas Solitones

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def simulate_nls_solitons():
    # Grid parameters
    N = 1024
    L = 40.0
    dt = L / N
    t = np.linspace(-L/2, L/2, N, endpoint=False)
    
    # Frequency grid
    omega = 2 * np.pi * np.fft.fftfreq(N, d=dt)
    
    # Initial condition: two colliding solitons
    # u(0, t) = sech(t+10)*exp(i*v1*t) + sech(t-10)*exp(i*v2*t)
    v1, v2 = 2.0, -2.0
    u0 = 1.2 * np.cosh(t + 10)**(-1) * np.exp(1j * v1 * t) + \
         1.2 * np.cosh(t - 10)**(-1) * np.exp(1j * v2 * t)
    
    u = u0.copy()
    
    # Propagation parameters
    dz = 0.01
    z_max = 10.0
    steps = int(z_max / dz)
    
    # Storage for plotting
    plot_steps = 100
    u_save = np.zeros((plot_steps, N), dtype=complex)
    save_interval = steps // plot_steps
    
    # Split-step Fourier Method (SSFM)
    # Linear operator: D = exp(-i * omega^2 * dz / 2)
    lin_oper = np.exp(-1j * (omega**2) * dz / 2.0)
    
    for i in range(steps):
        if i % save_interval == 0:
            u_save[i // save_interval] = u
        
        # Half step linear
        u = np.fft.ifft(np.fft.fft(u) * np.exp(-1j * (omega**2) * dz / 4.0))
        # Nonlinear step
        u = u * np.exp(1j * np.abs(u)**2 * dz)
        # Half step linear
        u = np.fft.ifft(np.fft.fft(u) * np.exp(-1j * (omega**2) * dz / 4.0))
        
    # Plotting
    Z, T = np.meshgrid(t, np.linspace(0, z_max, plot_steps))
    
    plt.figure(figsize=(10, 6))
    plt.pcolormesh(T, Z, np.abs(u_save)**2, shading='auto', cmap='inferno')
    plt.colorbar(label='|u|^2 (Intensity)')
    plt.title('Nonlinear Schrödinger Equation: Soliton Collision')
    plt.xlabel('Time (t)')
    plt.ylabel('Distance (z)')
    plt.tight_layout()
    plt.savefig('Ondas_Solitones_Collision.png')
    plt.show()

if __name__ == '__main__':
    simulate_nls_solitons()
